# Training Notebook — Synthetic Shapes Dataset

Generates a synthetic instance segmentation dataset (circles, rectangles, triangles), registers it with Detectron2, and runs training to validate the full pipeline.

In [ ]:
import sys
import json
import os
import random
import math
from pathlib import Path

import cv2
import numpy as np

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from training_scripts.train import register_instances, train

## 1. Generate Synthetic Shapes Dataset

In [ ]:
CATEGORY_INFO = [
    {"id": 0, "name": "circle"},
    {"id": 1, "name": "rectangle"},
    {"id": 2, "name": "triangle"},
]

def _random_color():
    """Bright, saturated color that stands out on a dark background."""
    return tuple(random.randint(80, 255) for _ in range(3))

def _draw_circle(img, ann_id, image_id):
    h, w = img.shape[:2]
    r = random.randint(30, 80)
    cx = random.randint(r + 5, w - r - 5)
    cy = random.randint(r + 5, h - r - 5)
    color = _random_color()
    cv2.circle(img, (cx, cy), r, color, -1)
    # polygon approximation (32 vertices)
    pts = []
    for i in range(32):
        angle = 2 * math.pi * i / 32
        pts.extend([cx + r * math.cos(angle), cy + r * math.sin(angle)])
    x0, y0 = cx - r, cy - r
    return {
        "id": ann_id,
        "image_id": image_id,
        "category_id": 0,
        "segmentation": [pts],
        "bbox": [float(x0), float(y0), float(2 * r), float(2 * r)],
        "area": float(math.pi * r * r),
        "iscrowd": 0,
    }

def _draw_rectangle(img, ann_id, image_id):
    h, w = img.shape[:2]
    rw = random.randint(40, 140)
    rh = random.randint(40, 140)
    x0 = random.randint(5, w - rw - 5)
    y0 = random.randint(5, h - rh - 5)
    color = _random_color()
    cv2.rectangle(img, (x0, y0), (x0 + rw, y0 + rh), color, -1)
    pts = [
        x0, y0,
        x0 + rw, y0,
        x0 + rw, y0 + rh,
        x0, y0 + rh,
    ]
    return {
        "id": ann_id,
        "image_id": image_id,
        "category_id": 1,
        "segmentation": [[float(p) for p in pts]],
        "bbox": [float(x0), float(y0), float(rw), float(rh)],
        "area": float(rw * rh),
        "iscrowd": 0,
    }

def _draw_triangle(img, ann_id, image_id):
    h, w = img.shape[:2]
    cx = random.randint(80, w - 80)
    cy = random.randint(80, h - 80)
    size = random.randint(40, 80)
    # equilateral-ish triangle
    pts_np = np.array([
        [cx, cy - size],
        [cx - size, cy + size],
        [cx + size, cy + size],
    ], dtype=np.int32)
    color = _random_color()
    cv2.fillPoly(img, [pts_np], color)
    pts = pts_np.flatten().tolist()
    xs, ys = pts_np[:, 0], pts_np[:, 1]
    x0, y0 = int(xs.min()), int(ys.min())
    bw, bh = int(xs.max() - x0), int(ys.max() - y0)
    return {
        "id": ann_id,
        "image_id": image_id,
        "category_id": 2,
        "segmentation": [[float(p) for p in pts]],
        "bbox": [float(x0), float(y0), float(bw), float(bh)],
        "area": float(0.5 * bw * bh),
        "iscrowd": 0,
    }

DRAW_FNS = [_draw_circle, _draw_rectangle, _draw_triangle]

def generate_split(out_dir, split_name, num_images, img_size=512, seed=42):
    """Generate one split (train/val/test) of synthetic shapes."""
    random.seed(seed)
    np.random.seed(seed)

    split_dir = os.path.join(out_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    images_info = []
    annotations = []
    ann_id = 0

    for img_idx in range(num_images):
        img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        # add slight background noise
        img += np.random.randint(0, 30, img.shape, dtype=np.uint8)

        num_shapes = random.randint(2, 6)
        for _ in range(num_shapes):
            draw_fn = random.choice(DRAW_FNS)
            ann = draw_fn(img, ann_id, img_idx)
            annotations.append(ann)
            ann_id += 1

        fname = f"{split_name}_{img_idx:04d}.png"
        cv2.imwrite(os.path.join(split_dir, fname), img)
        images_info.append({
            "id": img_idx,
            "file_name": fname,
            "width": img_size,
            "height": img_size,
        })

    coco = {
        "images": images_info,
        "annotations": annotations,
        "categories": CATEGORY_INFO,
    }
    ann_path = os.path.join(split_dir, f"{split_name}.json")
    with open(ann_path, "w") as f:
        json.dump(coco, f)

    print(f"  {split_name}: {num_images} images, {len(annotations)} annotations -> {split_dir}")
    return ann_path

In [ ]:
DATASET_DIR = str(PROJECT_ROOT / "synthetic_shapes")

print("Generating synthetic shapes dataset...")
generate_split(DATASET_DIR, "train", num_images=50, seed=42)
generate_split(DATASET_DIR, "val",   num_images=15, seed=123)
generate_split(DATASET_DIR, "test",  num_images=15, seed=456)
print("Done!")

## 2. Register Dataset & Train

In [ ]:
from detectron2 import model_zoo

# Register the synthetic dataset splits
register_instances(DATASET_DIR)

BASE_MODEL = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

config = {
    "BASE_MODEL": BASE_MODEL,
    "JOBNAME": "synthetic_shapes_test",
    "EVAL_PERIOD": 100,

    "MODEL": {
        "WEIGHTS": model_zoo.get_checkpoint_url(BASE_MODEL),
        "ROI_HEADS": {
            "NUM_CLASSES": 3,
            "BATCH_SIZE_PER_IMAGE": 128,
            "SCORE_THRESH_TEST": 0.3,
        },
    },

    "DATASETS": {
        "TRAIN": ["my_dataset_train"],
        "TEST": ["my_dataset_test", "my_dataset_val"],
    },

    "DATALOADER": {
        "NUM_WORKERS": 2,
        "FILTER_EMPTY_ANNOTATIONS": False,
    },

    "SOLVER": {
        "MAX_ITER": 300,
        "IMS_PER_BATCH": 2,
        "BASE_LR": 0.001,
        "STEPS": [200, 250],
        "GAMMA": 0.5,
        "WARMUP_ITERS": 50,
        "WARMUP_FACTOR": 0.1,
    },

    "OUTPUT_DIR": "model_output/",
}

train(config)